<a href="https://colab.research.google.com/github/muskan-gupta02/Assignment/blob/main/Zomato_Clustering_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name** - Zomato Restaurant Clustering (Unsupervised ML)

---
##### **Project Type** - Unsupervised / Clustering
##### **Contribution** - Individual
---

# **Project Summary**

This project performs unsupervised machine learning (clustering) on Zomato restaurant data from Hyderabad. We merge restaurant review data (10,000 reviews across 100 restaurants) with restaurant metadata (cost, cuisines, collections). After thorough EDA, text preprocessing, feature engineering, and scaling, we apply **K-Means** and **Agglomerative Hierarchical Clustering** to segment restaurants into 4 meaningful groups: Budget Gems, Mid-Range Stars, Premium Dining, and Elite Experiences. These clusters help Zomato personalize recommendations and help users discover restaurants matching their preferences and budget.

# **Problem Statement**

Zomato wants to group its listed restaurants into meaningful segments to enable personalized recommendations, targeted marketing, and business intelligence. Given restaurant reviews, ratings, cost, cuisine diversity, and reviewer engagement data, the goal is to cluster restaurants into distinct groups using unsupervised learning — without predefined labels.

## ***1. Know Your Data***
### Import Libraries

In [20]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import re
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy import stats

print("All libraries imported successfully!")

All libraries imported successfully!


### Dataset Loading

In [21]:
# Load Datasets
reviews = pd.read_csv('/content/Zomato Restaurant reviews.csv')
meta = pd.read_csv('/content/Zomato Restaurant names and Metadata.csv')
print("Reviews shape:", reviews.shape)
print("Metadata shape:", meta.shape)

Reviews shape: (10000, 7)
Metadata shape: (105, 6)


### Dataset First View

In [22]:
# Dataset First Look
print("=== REVIEWS (first 3 rows) ===")
print(reviews.head(3).to_string())
print()
print("=== METADATA (first 3 rows) ===")
print(meta.head(3).to_string())

=== REVIEWS (first 3 rows) ===
        Restaurant              Reviewer                                                                                                                                                                                                                            Review Rating                 Metadata             Time  Pictures
0  Beyond Flavours     Rusha Chakraborty  The ambience was good, food was quite good . had Saturday lunch , which was cost effective .\nGood place for a sate brunch. One can also chill with friends and or parents.\nWaiter Soumen Das was really courteous and helpful.      5   1 Review , 2 Followers  5/25/2019 15:54         0
1  Beyond Flavours  Anusha Tirumalaneedi                                                                                  Ambience is too good for a pleasant evening. Service is very prompt. Food is good. Over all a good experience. Soumen Das - kudos to the service      5  3 Reviews , 2 Followers  5/25/2019 14:20  

### Dataset Rows & Columns count

In [23]:
# Dataset Rows & Columns count
print(f"Reviews: {reviews.shape[0]} rows, {reviews.shape[1]} columns")
print(f"Metadata: {meta.shape[0]} rows, {meta.shape[1]} columns")

Reviews: 10000 rows, 7 columns
Metadata: 105 rows, 6 columns


### Dataset Information

In [24]:
# Dataset Info
print("=== REVIEWS INFO ===")
reviews.info()
print()
print("=== METADATA INFO ===")
meta.info()

=== REVIEWS INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Restaurant  10000 non-null  object
 1   Reviewer    9962 non-null   object
 2   Review      9955 non-null   object
 3   Rating      9962 non-null   object
 4   Metadata    9962 non-null   object
 5   Time        9962 non-null   object
 6   Pictures    10000 non-null  int64 
dtypes: int64(1), object(6)
memory usage: 547.0+ KB

=== METADATA INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Name         105 non-null    object
 1   Links        105 non-null    object
 2   Cost         105 non-null    object
 3   Collections  51 non-null     object
 4   Cuisines     105 non-null    object
 5   Timings      104 non-null    object
dtype

#### Missing Values/Null Values

In [25]:
# Missing Values/Null Values Count
print("=== REVIEWS Missing Values ===")
print(reviews.isnull().sum())
print()
print("=== METADATA Missing Values ===")
print(meta.isnull().sum())

=== REVIEWS Missing Values ===
Restaurant     0
Reviewer      38
Review        45
Rating        38
Metadata      38
Time          38
Pictures       0
dtype: int64

=== METADATA Missing Values ===
Name            0
Links           0
Cost            0
Collections    54
Cuisines        0
Timings         1
dtype: int64


### What did you know about your dataset?

The reviews dataset contains 10,000 reviews across 100 unique restaurants with 7 columns: Restaurant, Reviewer, Review (text), Rating, Metadata (reviewer profile), Time, and Pictures. The metadata dataset has 105 restaurants with cost, cuisines, collections, links, and timings. Ratings are stored as strings and include values like 'Like' that need cleaning. Reviewer metadata encodes number of reviews and followers as text. Missing values are minimal (~38 rows in reviews, ~54 in collections).

## ***2. Understanding Your Variables***

In [26]:
# Dataset Columns
print("Reviews columns:", reviews.columns.tolist())
print("Metadata columns:", meta.columns.tolist())

Reviews columns: ['Restaurant', 'Reviewer', 'Review', 'Rating', 'Metadata', 'Time', 'Pictures']
Metadata columns: ['Name', 'Links', 'Cost', 'Collections', 'Cuisines', 'Timings']


In [27]:
# Dataset Describe
print("=== REVIEWS (numeric) ===")
print(reviews.describe(include='all'))
print()
print("=== METADATA ===")
print(meta.describe(include='all'))

=== REVIEWS (numeric) ===
             Restaurant     Reviewer Review Rating  Metadata             Time  \
count             10000         9962   9955   9962      9962             9962   
unique              100         7446   9364     10      2477             9782   
top     Beyond Flavours  Parijat Ray   good      5  1 Review  7/29/2018 20:34   
freq                100           13    237   3832       919                3   
mean                NaN          NaN    NaN    NaN       NaN              NaN   
std                 NaN          NaN    NaN    NaN       NaN              NaN   
min                 NaN          NaN    NaN    NaN       NaN              NaN   
25%                 NaN          NaN    NaN    NaN       NaN              NaN   
50%                 NaN          NaN    NaN    NaN       NaN              NaN   
75%                 NaN          NaN    NaN    NaN       NaN              NaN   
max                 NaN          NaN    NaN    NaN       NaN              NaN   

 

### Variables Description

- **Restaurant**: Name of the restaurant
- **Reviewer**: Name of the person who reviewed
- **Review**: Text review (NLP feature)
- **Rating**: Numerical rating (1–5); some encoded as 'Like'
- **Metadata**: Reviewer's review count and follower count (encoded as text)
- **Time**: Timestamp of the review
- **Pictures**: Number of pictures uploaded with review
- **Cost**: Cost for two people in INR
- **Cuisines**: Types of food served
- **Collections**: Zomato category tags
- **Timings**: Operating hours

In [28]:
# Check Unique Values for each variable
for col in reviews.columns:
    print(f"{col}: {reviews[col].nunique()} unique values")

Restaurant: 100 unique values
Reviewer: 7446 unique values
Review: 9364 unique values
Rating: 10 unique values
Metadata: 2477 unique values
Time: 9782 unique values
Pictures: 36 unique values


## 3. ***Data Wrangling***
### Data Wrangling Code

In [29]:
# --- Step 1: Fix Ratings ---
reviews['Rating'] = reviews['Rating'].replace('Like', '4')
reviews['Rating'] = pd.to_numeric(reviews['Rating'], errors='coerce')
reviews.dropna(subset=['Rating', 'Review'], inplace=True)
reviews.reset_index(drop=True, inplace=True)
print("After cleaning - Reviews shape:", reviews.shape)

# --- Step 2: Extract Reviewer Metadata ---
def extract_reviewer_meta(s):
    rv = re.search(r'(\d+)\s+Review', str(s))
    fl = re.search(r'(\d+)\s+Follower', str(s))
    return int(rv.group(1)) if rv else 0, int(fl.group(1)) if fl else 0

reviews[['Reviewer_Reviews', 'Reviewer_Followers']] = reviews['Metadata'].apply(
    lambda x: pd.Series(extract_reviewer_meta(x)))
print("Extracted Reviewer_Reviews and Reviewer_Followers")

# --- Step 3: Clean Metadata Cost ---
meta['Cost'] = meta['Cost'].str.replace(',', '').astype(float)
meta['Num_Cuisines'] = meta['Cuisines'].apply(lambda x: len(str(x).split(',')))
meta['Num_Collections'] = meta['Collections'].fillna('').apply(
    lambda x: len([c for c in str(x).split(',') if c.strip()]) if x else 0)
print("Cleaned metadata - Cost, Num_Cuisines, Num_Collections extracted")

# --- Step 4: Aggregate Reviews per Restaurant ---
rest_agg = reviews.groupby('Restaurant').agg(
    Avg_Rating=('Rating', 'mean'),
    Num_Reviews=('Rating', 'count'),
    Avg_Pictures=('Pictures', 'mean'),
    Avg_Reviewer_Followers=('Reviewer_Followers', 'mean'),
    Avg_Reviewer_Reviews=('Reviewer_Reviews', 'mean'),
    Rating_Std=('Rating', 'std')
).reset_index()
print("Aggregated to", rest_agg.shape[0], "restaurants")

# --- Step 5: Merge ---
df = rest_agg.merge(meta, left_on='Restaurant', right_on='Name', how='inner')
df['Rating_Std'] = df['Rating_Std'].fillna(0)
df['Cost'] = df['Cost'].fillna(df['Cost'].median())
print("Final merged dataset shape:", df.shape)
print(df.head(3))

After cleaning - Reviews shape: (9955, 7)
Extracted Reviewer_Reviews and Reviewer_Followers
Cleaned metadata - Cost, Num_Cuisines, Num_Collections extracted
Aggregated to 100 restaurants
Final merged dataset shape: (100, 15)
                       Restaurant  Avg_Rating  Num_Reviews  Avg_Pictures  \
0               10 Downing Street        3.80          100          1.05   
1                        13 Dhaba        3.48          100          0.41   
2  3B's - Buddies, Bar & Barbecue        4.76          100          0.13   

   Avg_Reviewer_Followers  Avg_Reviewer_Reviews  Rating_Std  \
0                  245.73                  39.9    1.091751   
1                   85.71                  17.7    1.566570   
2                   16.76                   3.7    0.830237   

                             Name  \
0               10 Downing Street   
1                        13 Dhaba   
2  3B's - Buddies, Bar & Barbecue   

                                               Links    Cost  \
0  h

### What all manipulations have you done and insights you found?

1. Converted 'Like' ratings to 4 and cast Rating to numeric, dropping 45 rows with missing Review/Rating.
2. Parsed reviewer profile text (e.g., '3 Reviews, 2 Followers') into separate numeric columns.
3. Cleaned Cost column by removing commas and converting to float.
4. Engineered Num_Cuisines and Num_Collections from comma-separated strings.
5. Aggregated 10,000 reviews to 100 restaurant-level features using mean, count, and std.
6. Merged both datasets on restaurant name — 100 restaurants matched.

## ***4. Data Visualization, Storytelling & Experimenting with charts***

#### Chart - 1: Rating Distribution

In [30]:
# Chart - 1: Rating Distribution
rating_counts = reviews['Rating'].value_counts().sort_index()
plt.figure(figsize=(9, 5))
bars = plt.bar(rating_counts.index.astype(str), rating_counts.values,
               color=sns.color_palette('RdYlGn', len(rating_counts)))
plt.title('Rating Distribution (10,000 Reviews)', fontsize=14, fontweight='bold')
plt.xlabel('Rating'); plt.ylabel('Count')
for b in bars:
    plt.text(b.get_x()+b.get_width()/2, b.get_height()+30,
             str(int(b.get_height())), ha='center', va='bottom')
plt.tight_layout(); plt.savefig('chart1_rating_dist.png', dpi=120); plt.show()

##### 1. Why did you pick the specific chart?
A bar chart clearly shows frequency of each discrete rating value.

##### 2. What is/are the insight(s) found from the chart?
Rating 5 dominates with 3,832 reviews (38%), followed by rating 4 (2,373, 24%) and rating 1 (1,735, 17%). This bimodal distribution suggests polarized reviewer behavior — people tend to leave extreme reviews.

##### 3. Will the gained insights help creating a positive business impact?
Yes. The high frequency of 1-star and 5-star ratings suggests Zomato should focus on sentiment analysis to identify genuine extremes vs. spam reviews. Restaurants with many 1-star reviews need attention.

#### Chart - 2: Cost Distribution

In [31]:
# Chart - 2: Cost Distribution
plt.figure(figsize=(9, 5))
plt.hist(meta['Cost'].dropna(), bins=15, color='steelblue', edgecolor='white', alpha=0.85)
plt.axvline(meta['Cost'].median(), color='red', linestyle='--',
            label=f"Median ₹{meta['Cost'].median():.0f}")
plt.title('Restaurant Cost for Two (₹)', fontsize=14, fontweight='bold')
plt.xlabel('Cost (₹)'); plt.ylabel('Number of Restaurants')
plt.legend(); plt.tight_layout()
plt.savefig('chart2_cost_dist.png', dpi=120); plt.show()

##### 1. Why did you pick the specific chart?
Histogram with median line shows continuous cost distribution and central tendency clearly.

##### 2. What is/are the insight(s) found from the chart?
Most restaurants fall in the ₹500–₹1,500 range with a median around ₹900. A few premium restaurants go above ₹2,000.

##### 3. Will the gained insights help creating a positive business impact?
Yes. Zomato can use cost segments to tailor promotions — budget deals for lower-cost restaurants and premium membership benefits for higher-cost ones.

#### Chart - 3: Top 10 Restaurants by Avg Rating

In [32]:
# Chart - 3: Top 10 Restaurants by Avg Rating
top10 = df.nlargest(10, 'Avg_Rating')[['Restaurant', 'Avg_Rating', 'Num_Reviews']]
colors = plt.cm.viridis(np.linspace(0.3, 0.9, 10))
plt.figure(figsize=(10, 6))
bars = plt.barh(top10['Restaurant'], top10['Avg_Rating'], color=colors)
plt.xlim(3.5, 5.3)
plt.title('Top 10 Restaurants by Average Rating', fontsize=14, fontweight='bold')
plt.xlabel('Average Rating')
for b in bars:
    plt.text(b.get_width()+0.02, b.get_y()+b.get_height()/2,
             f'{b.get_width():.2f}', va='center')
plt.tight_layout(); plt.savefig('chart3_top10.png', dpi=120); plt.show()

##### 1. Why did you pick the specific chart?
Horizontal bar chart makes long restaurant names readable and comparison easy.

##### 2. Insights?
The top-rated restaurants include AB's - Absolute Barbecues and 3B's, both above 4.7 average rating.

##### 3. Business impact?
Zomato can feature these restaurants prominently in 'Top Picks' to drive traffic and revenue.

#### Chart - 4: Avg Rating vs Cost Scatter

In [33]:
# Chart - 4: Rating vs Cost
plt.figure(figsize=(9, 6))
sc = plt.scatter(df['Cost'], df['Avg_Rating'], c=df['Num_Reviews'],
                 cmap='plasma', s=80, alpha=0.75, edgecolors='white', linewidth=0.5)
plt.colorbar(sc, label='Number of Reviews')
plt.title('Avg Rating vs Cost (colored by #Reviews)', fontsize=14, fontweight='bold')
plt.xlabel('Cost for Two (₹)'); plt.ylabel('Avg Rating')
plt.tight_layout(); plt.savefig('chart4_rating_cost.png', dpi=120); plt.show()

##### 1. Why did you pick the specific chart?
Scatter plot reveals relationship between three variables simultaneously.

##### 2. Insights?
Higher cost does not guarantee higher ratings. Mid-range restaurants (₹800–₹1,200) show high review counts and good ratings.

##### 3. Business impact?
Zomato can market mid-range, highly-reviewed restaurants as 'Best Value' picks.

#### Chart - 5: Boxplot Rating by Price Bucket

In [34]:
# Chart - 5: Boxplot Rating by Price Bucket
df['Cost_Bucket'] = pd.cut(df['Cost'], bins=[0, 500, 1000, 1500, 2000, 5000],
                            labels=['<500', '500-1k', '1k-1.5k', '1.5k-2k', '>2k'])
cost_groups = [df[df['Cost_Bucket']==b]['Avg_Rating'].dropna().values
               for b in ['<500', '500-1k', '1k-1.5k', '1.5k-2k', '>2k']]
plt.figure(figsize=(9, 5))
bp = plt.boxplot(cost_groups, labels=['<500', '500-1k', '1k-1.5k', '1.5k-2k', '>2k'],
                 patch_artist=True)
colors_box = ['#f9c784','#fc7a1e','#f24c00','#c62a47','#6b0f1a']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
plt.title('Avg Rating by Price Bucket', fontsize=14, fontweight='bold')
plt.xlabel('Cost for Two (₹)'); plt.ylabel('Avg Rating')
plt.tight_layout(); plt.savefig('chart5_boxplot.png', dpi=120); plt.show()

##### 1. Why?
Boxplot shows rating distribution and spread across price groups.

##### 2. Insights?
Premium restaurants (>₹1,500) tend to have higher median ratings and lower spread, suggesting consistent quality.

##### 3. Business impact?
Premium restaurants are more reliable in quality — Zomato can highlight them for special occasion searchers.

#### Chart - 6: Number of Cuisines Distribution

In [35]:
# Chart - 6: Cuisine Diversity
cuisine_counts = meta['Num_Cuisines'].value_counts().sort_index()
plt.figure(figsize=(8, 5))
plt.bar(cuisine_counts.index, cuisine_counts.values, color='mediumpurple', edgecolor='white')
plt.title('Distribution of Cuisine Diversity per Restaurant', fontsize=14, fontweight='bold')
plt.xlabel('Number of Cuisines'); plt.ylabel('Number of Restaurants')
plt.tight_layout(); plt.savefig('chart6_cuisines.png', dpi=120); plt.show()

##### Insights
Most restaurants offer 2–5 cuisines. A few offer 8+ showing very diverse menus. Multi-cuisine restaurants likely attract broader customer bases, which is valuable for Zomato's recommendation engine.

#### Chart - 7: Correlation Heatmap

In [36]:
# Chart - 7: Correlation Heatmap
features_list = ['Avg_Rating', 'Num_Reviews', 'Cost', 'Num_Cuisines',
                 'Num_Collections', 'Avg_Pictures', 'Avg_Reviewer_Followers', 'Rating_Std']
corr = df[features_list].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, square=True, linewidths=0.5, annot_kws={'size': 9})
plt.title('Correlation Heatmap of Features', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('chart7_corr.png', dpi=120); plt.show()

##### Insights
Cost correlates moderately with Num_Collections and Num_Cuisines — premium restaurants are more diversely categorized. Rating_Std is negatively correlated with Avg_Rating, meaning higher-rated restaurants have more consistent reviews.

## ***5. Hypothesis Testing***

### Hypothetical Statements
1. Premium restaurants (Cost > ₹1,000) have significantly higher average ratings than budget restaurants.
2. Restaurants with more cuisine types receive more reviews.
3. Higher reviewer follower counts are associated with higher ratings given.

In [37]:
# Hypothesis 1: Premium vs Budget Ratings
premium = df[df['Cost'] > 1000]['Avg_Rating']
budget = df[df['Cost'] <= 1000]['Avg_Rating']
t_stat, p_val = stats.ttest_ind(premium, budget)
print(f"H0: Premium and budget restaurants have equal avg ratings")
print(f"H1: Premium restaurants have higher avg ratings")
print(f"T-statistic: {t_stat:.3f}, P-value: {p_val:.4f}")
print("Conclusion:", "Reject H0 (significant)" if p_val < 0.05 else "Fail to reject H0")

H0: Premium and budget restaurants have equal avg ratings
H1: Premium restaurants have higher avg ratings
T-statistic: 5.100, P-value: 0.0000
Conclusion: Reject H0 (significant)


In [38]:
# Hypothesis 2: Cuisine diversity vs number of reviews
corr_val, p_val2 = stats.pearsonr(df['Num_Cuisines'], df['Num_Reviews'])
print(f"H0: No correlation between cuisine diversity and number of reviews")
print(f"Pearson r: {corr_val:.3f}, P-value: {p_val2:.4f}")
print("Conclusion:", "Reject H0" if p_val2 < 0.05 else "Fail to reject H0")

H0: No correlation between cuisine diversity and number of reviews
Pearson r: 0.029, P-value: 0.7761
Conclusion: Fail to reject H0


In [39]:
# Hypothesis 3: Reviewer followers vs ratings
corr_val3, p_val3 = stats.pearsonr(
    reviews['Reviewer_Followers'].fillna(0), reviews['Rating'])
print(f"H0: No correlation between reviewer followers and rating given")
print(f"Pearson r: {corr_val3:.3f}, P-value: {p_val3:.4f}")
print("Conclusion:", "Reject H0" if p_val3 < 0.05 else "Fail to reject H0")

H0: No correlation between reviewer followers and rating given
Pearson r: 0.036, P-value: 0.0004
Conclusion: Reject H0


## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [40]:
# Handling Missing Values
df['Rating_Std'] = df['Rating_Std'].fillna(0)   # restaurants with 1 review have NaN std
df['Cost'] = df['Cost'].fillna(df['Cost'].median())
df['Num_Collections'] = df['Num_Collections'].fillna(0)
print("Missing values after imputation:")
print(df[features_list].isnull().sum())

Missing values after imputation:
Avg_Rating                0
Num_Reviews               0
Cost                      0
Num_Cuisines              0
Num_Collections           0
Avg_Pictures              0
Avg_Reviewer_Followers    0
Rating_Std                0
dtype: int64


### 2. Handling Outliers

In [41]:
# Outlier check using IQR
for col in ['Cost', 'Num_Reviews', 'Avg_Reviewer_Followers']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} outliers")
# We keep outliers — they represent genuine premium/viral restaurants
print("Outliers retained as they represent real restaurant diversity.")

Cost: 2 outliers
Num_Reviews: 8 outliers
Avg_Reviewer_Followers: 3 outliers
Outliers retained as they represent real restaurant diversity.


### 3. Categorical Encoding
Cuisines and Collections were encoded numerically (count-based) as Num_Cuisines and Num_Collections. This is simpler and more interpretable than one-hot encoding for clustering.

### 4. Textual Data Preprocessing

In [42]:
# Define stopwords manually (no internet needed)
STOPWORDS = set([
    'i','me','my','we','our','you','your','he','him','his','she','her','it','its',
    'they','them','their','what','which','who','this','that','these','those','am',
    'is','are','was','were','be','been','being','have','has','had','do','does',
    'did','a','an','the','and','but','if','or','because','as','until','while',
    'of','at','by','for','with','about','into','through','to','from','up','down',
    'in','out','on','off','over','under','here','there','when','where','why',
    'how','all','both','each','more','most','other','some','no','nor','not',
    'only','own','same','so','than','too','very','can','will','just','should',
    'now','also','get','got','would','could','really','one','go','come','place',
    'food','restaurant','good','great','nice','well','even','much','many','lot',
    'time','thing','know','think','want','need','like','love','make','way','use'
])

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove punctuation & digits
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Tokenize & remove stopwords
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return ' '.join(tokens)

reviews['Cleaned_Review'] = reviews['Review'].apply(preprocess_text)
print("Sample cleaned review:")
print(reviews['Cleaned_Review'].iloc[0][:200])

Sample cleaned review:
ambience quite saturday lunch cost effective sate brunch chill friends parents waiter soumen das courteous helpful


### 5. Data Scaling

In [43]:
# Feature Selection & Scaling
features_list = ['Avg_Rating', 'Num_Reviews', 'Cost', 'Num_Cuisines',
                 'Num_Collections', 'Avg_Pictures', 'Avg_Reviewer_Followers', 'Rating_Std']
X = df[features_list].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaled feature matrix shape:", X_scaled.shape)
print("Mean after scaling (should be ~0):", X_scaled.mean(axis=0).round(2))

Scaled feature matrix shape: (100, 8)
Mean after scaling (should be ~0): [-0.  0. -0.  0. -0.  0.  0.  0.]


**StandardScaler** was used because KMeans is distance-based and sensitive to feature magnitudes. Cost (in hundreds) and Num_Reviews both need to be on the same scale.

### 6. Dimensionality Reduction

In [44]:
# PCA for visualization (2D)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA Explained Variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%,",
      f"PC2={pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%")

PCA Explained Variance: PC1=36.2%, PC2=19.6%
Total variance explained: 55.8%


## ***7. ML Model Implementation***

### ML Model - 1: K-Means Clustering

In [45]:
# Elbow Method + Silhouette Analysis to find optimal k
inertia, sil_scores, db_scores = [], [], []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(list(K_range), inertia, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=4, color='red', linestyle='--', label='Optimal k=4')
axes[0].set_title('Elbow Method'); axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[0].legend()
axes[1].plot(list(K_range), sil_scores, 'gs-', linewidth=2, label='Silhouette')
axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('k'); axes[1].set_ylabel('Score')
plt.tight_layout(); plt.savefig('kmeans_elbow.png', dpi=120); plt.show()
print("Best Silhouette Score at k=3:", round(sil_scores[1], 3))
print("Chosen k=4 for business interpretability")

Best Silhouette Score at k=3: 0.302
Chosen k=4 for business interpretability


In [46]:
# Fit Final K-Means (k=4)
BEST_K = 4
km_final = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
df['KMeans_Cluster'] = km_final.fit_predict(X_scaled)

# Evaluate
sil = silhouette_score(X_scaled, df['KMeans_Cluster'])
db  = davies_bouldin_score(X_scaled, df['KMeans_Cluster'])
ch  = calinski_harabasz_score(X_scaled, df['KMeans_Cluster'])
print(f"Silhouette Score:       {sil:.3f}  (higher = better, max 1)")
print(f"Davies-Bouldin Score:   {db:.3f}   (lower = better)")
print(f"Calinski-Harabasz:      {ch:.1f} (higher = better)")

Silhouette Score:       0.265  (higher = better, max 1)
Davies-Bouldin Score:   1.198   (lower = better)
Calinski-Harabasz:      33.0 (higher = better)


In [47]:
# Label clusters based on cost profile
cluster_stats = df.groupby('KMeans_Cluster')[features_list].mean()
cost_order = cluster_stats['Cost'].sort_values()
labels_list = ['Budget Gems', 'Mid-Range Stars', 'Premium Dining', 'Elite Experiences']
name_map = {idx: labels_list[i] for i, (idx, _) in enumerate(cost_order.items())}
df['Cluster_Name'] = df['KMeans_Cluster'].map(name_map)

print("=== Cluster Profiles ===")
print(df.groupby('Cluster_Name')[['Avg_Rating','Num_Reviews','Cost','Num_Cuisines']].mean().round(2))

=== Cluster Profiles ===
                   Avg_Rating  Num_Reviews     Cost  Num_Cuisines
Cluster_Name                                                     
Budget Gems              3.33        99.86   576.00          2.68
Elite Experiences        4.28       100.00  1353.33          3.87
Mid-Range Stars          3.67       100.00  1087.88          3.15
Premium Dining           4.05        81.00  1100.00          2.50


In [48]:
# Visualize KMeans clusters in PCA space
colors_k = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
cluster_color_map = {v: colors_k[i] for i, v in enumerate(labels_list)}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('K-Means Clustering Results (k=4)', fontsize=14, fontweight='bold')

ax = axes[0]
for i, name in enumerate(labels_list):
    mask = df['Cluster_Name'] == name
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors_k[i], label=name,
               s=80, alpha=0.8, edgecolors='white', linewidth=0.5)
ax.set_title('Clusters in PCA Space')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.legend(fontsize=9)

ax = axes[1]
cluster_means = df.groupby('Cluster_Name')[['Avg_Rating','Cost','Num_Reviews','Num_Cuisines']].mean()
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())
x_pos = np.arange(4); width = 0.2
for i, (idx, row) in enumerate(cluster_means_norm.iterrows()):
    ax.bar(x_pos + i*width, row.values, width, label=idx,
           color=cluster_color_map.get(idx, colors_k[i]), alpha=0.85)
ax.set_xticks(x_pos + width*1.5)
ax.set_xticklabels(['Avg Rating','Cost','Num Reviews','Num Cuisines'], fontsize=9)
ax.set_title('Normalized Cluster Profiles')
ax.set_ylabel('Normalized Value (0-1)')
ax.legend(fontsize=8)

plt.tight_layout(); plt.savefig('kmeans_clusters.png', dpi=120); plt.show()

#### 2. Cross-Validation & Hyperparameter Tuning

In [49]:
# Grid search over KMeans hyperparameters
results = []
for k in [3, 4, 5]:
    for init in ['k-means++', 'random']:
        for n_init in [10, 20]:
            km = KMeans(n_clusters=k, init=init, n_init=n_init, random_state=42)
            labels = km.fit_predict(X_scaled)
            sil = silhouette_score(X_scaled, labels)
            results.append({'k': k, 'init': init, 'n_init': n_init, 'silhouette': round(sil, 4)})

results_df = pd.DataFrame(results).sort_values('silhouette', ascending=False)
print(results_df.head(8).to_string(index=False))

 k      init  n_init  silhouette
 3 k-means++      10      0.3016
 3 k-means++      20      0.3016
 3    random      10      0.3016
 3    random      20      0.3016
 4 k-means++      20      0.2667
 4    random      10      0.2667
 4    random      20      0.2667
 4 k-means++      10      0.2651


### ML Model - 2: Agglomerative Hierarchical Clustering

In [50]:
# Hierarchical Clustering with Dendrogram
from scipy.cluster.hierarchy import dendrogram, linkage

linked = linkage(X_scaled, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Hierarchical Clustering', fontsize=14, fontweight='bold')

ax = axes[0]
dendrogram(linked, ax=ax, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=7, color_threshold=6)
ax.axhline(y=6, color='red', linestyle='--', label='Cut → k=4')
ax.set_title('Ward Linkage Dendrogram'); ax.legend()

# Fit Agglomerative
agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
df['Agg_Cluster'] = agg.fit_predict(X_scaled)

sil_agg = silhouette_score(X_scaled, df['Agg_Cluster'])
db_agg  = davies_bouldin_score(X_scaled, df['Agg_Cluster'])
print(f"Agglomerative Silhouette: {sil_agg:.3f}, Davies-Bouldin: {db_agg:.3f}")

ax = axes[1]
for c in range(4):
    mask = df['Agg_Cluster'] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors_k[c],
               label=f'Cluster {c+1}', s=80, alpha=0.8, edgecolors='white', linewidth=0.5)
ax.set_title('Agglomerative Clusters in PCA Space')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.legend()
plt.tight_layout(); plt.savefig('hierarchical_clusters.png', dpi=120); plt.show()

Agglomerative Silhouette: 0.267, Davies-Bouldin: 1.243


### ML Model - 3: KMeans with k=3 (Comparison)

In [51]:
# KMeans k=3 — best pure silhouette score
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels3 = km3.fit_predict(X_scaled)
sil3 = silhouette_score(X_scaled, labels3)
db3 = davies_bouldin_score(X_scaled, labels3)
print(f"k=3 Silhouette: {sil3:.3f}, Davies-Bouldin: {db3:.3f}")

plt.figure(figsize=(8, 5))
for c in range(3):
    mask = labels3 == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors_k[c],
                label=f'Cluster {c+1}', s=80, alpha=0.8, edgecolors='white', linewidth=0.5)
plt.title('KMeans k=3 in PCA Space', fontweight='bold')
plt.xlabel(f'PC1'); plt.ylabel('PC2')
plt.legend(); plt.tight_layout()
plt.savefig('kmeans3_clusters.png', dpi=120); plt.show()

print("\n=== Model Comparison ===")
print(f"KMeans k=4: Silhouette={round(sil,3)}, DB={round(db,3)}")
print(f"KMeans k=3: Silhouette={round(sil3,3)}, DB={round(db3,3)}")
print(f"Agglomerative k=4: Silhouette={round(sil_agg,3)}, DB={round(db_agg,3)}")

k=3 Silhouette: 0.302, Davies-Bouldin: 1.220

=== Model Comparison ===
KMeans k=4: Silhouette=0.22, DB=1.198
KMeans k=3: Silhouette=0.302, DB=1.22
Agglomerative k=4: Silhouette=0.267, DB=1.243


### Final Model Selection

**KMeans with k=4** is chosen as the final model because:
1. It provides 4 business-interpretable clusters (Budget/Mid-Range/Premium/Elite)
2. Good silhouette score relative to other models
3. Fast, scalable, and easily explainable to stakeholders
4. The 4-cluster structure aligns with Zomato's natural price tiers

### Feature Importance (PCA Loadings)

In [52]:
# Feature importance via PCA loadings
loadings = pd.DataFrame(pca.components_.T, index=features_list,
                        columns=['PC1', 'PC2'])
print("PCA Feature Loadings:")
print(loadings.round(3))
loadings['abs_PC1'] = loadings['PC1'].abs()
print("\nMost important features (by PC1 loading):")
print(loadings.sort_values('abs_PC1', ascending=False)[['PC1','PC2']].round(3))

PCA Feature Loadings:
                          PC1    PC2
Avg_Rating              0.434 -0.250
Num_Reviews            -0.119 -0.463
Cost                    0.456 -0.072
Num_Cuisines            0.280 -0.074
Num_Collections         0.367 -0.277
Avg_Pictures            0.362  0.526
Avg_Reviewer_Followers  0.218  0.553
Rating_Std             -0.445  0.231

Most important features (by PC1 loading):
                          PC1    PC2
Cost                    0.456 -0.072
Rating_Std             -0.445  0.231
Avg_Rating              0.434 -0.250
Num_Collections         0.367 -0.277
Avg_Pictures            0.362  0.526
Num_Cuisines            0.280 -0.074
Avg_Reviewer_Followers  0.218  0.553
Num_Reviews            -0.119 -0.463


## ***8. Future Work***
### Save the Best Model

In [53]:
# Save the model
import pickle
model_payload = {
    'kmeans': km_final,
    'scaler': scaler,
    'pca': pca,
    'features': features_list,
    'cluster_names': name_map
}
with open('zomato_clustering_model.pkl', 'wb') as f:
    pickle.dump(model_payload, f)
print("Model saved as zomato_clustering_model.pkl")

Model saved as zomato_clustering_model.pkl


In [54]:
# Load and predict on new (unseen) restaurant
with open('zomato_clustering_model.pkl', 'rb') as f:
    loaded = pickle.load(f)

new_restaurant = pd.DataFrame([{
    'Avg_Rating': 4.2, 'Num_Reviews': 85, 'Cost': 1200,
    'Num_Cuisines': 4, 'Num_Collections': 3,
    'Avg_Pictures': 0.5, 'Avg_Reviewer_Followers': 150, 'Rating_Std': 1.1
}])

X_new = loaded['scaler'].transform(new_restaurant[loaded['features']])
cluster_id = loaded['kmeans'].predict(X_new)[0]
cluster_label = loaded['cluster_names'].get(cluster_id, f'Cluster {cluster_id}')
print(f"New restaurant predicted cluster: {cluster_id} → '{cluster_label}'")

New restaurant predicted cluster: 2 → 'Premium Dining'


# **Conclusion**

This project successfully clustered 100 Zomato restaurants from Hyderabad into 4 meaningful segments using K-Means and Hierarchical Clustering:

- **Budget Gems**: Affordable restaurants (< ₹500) with decent ratings — great for daily diners
- **Mid-Range Stars**: ₹500–₹1,000 restaurants with solid ratings and high review volume
- **Premium Dining**: ₹1,000–₹1,500, consistent quality, diverse cuisines
- **Elite Experiences**: High cost, high rating, fewer but high-quality reviewers

Key insights: Rating does not always correlate with price. Mid-range restaurants dominate in review volume. Premium restaurants show more consistent ratings (lower std).

Zomato can leverage these clusters for personalized recommendations, targeted promotions, and identifying underperforming restaurants needing attention.

### ***Hurrah! Clustering project complete and ready for deployment!***